# 03 Detector**GeoFish-AI** · runs top to bottom on a Kaggle GPU (T4 x2 or P100).**Before you run**1. *Settings → Accelerator*: **GPU T4 x2**2. *Settings → Internet*: **On** (pretrained weights are mandatory — the code raises rather than training from scratch)3. *Add Data*: attach the datasets listed in cell 3 (required ones are enforced)4. *Settings → Persistence*: **Files only**, so checkpoints survive between savesEvery stage writes into `/kaggle/working`, records itself in`/kaggle/working/outputs/logs/state.json`, and **skips itself on a re-run**.An interrupted session resumes from the last finished epoch/stage.

In [ ]:
# @title 0a · Enter the repo (clone once if this notebook was uploaded alone)import os, subprocess, sysfrom pathlib import PathREPO_DIR = Path("/kaggle/working/fishes")BRANCH = "arena/3dc076ac-fishes"def _is_repo(p): return (p / "configs" / "kaggle.yaml").exists()if _is_repo(Path.cwd()):    REPO = Path.cwd()elif _is_repo(REPO_DIR):    REPO = REPO_DIRelse:    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, "https://github.com/Cometbuster4969/fishes.git",                    str(REPO_DIR)], check=True)    REPO = REPO_DIRos.chdir(REPO)sys.path.insert(0, str(REPO))sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,                     text=True).stdout.strip()print("repo:", REPO, "| commit:", sha)

In [ ]:
# @title 0b · Dependencies (only what the Kaggle image lacks)import osON_KAGGLE = os.path.exists("/kaggle")if ON_KAGGLE:    %pip install -q -r requirements-kaggle.txt 2>&1 | tail -5else:    print("not on Kaggle: skipped")

In [ ]:
# @title 0c · Config, seeding, resume state, mount reportfrom src.utils.notebook import bootstrapfrom src.config import describeenv = bootstrap("03_detector", require_gpu=True)cfg, tr, log = env.cfg, env.tracker, env.logprint(describe(cfg))print()from src.utils.state import stage, validate_stateprint("stages already done:", [k for k, v in tr.state["stages"].items()                               if v.get("status") == "done"])

## 1 · Single-class detection datasetRebuilt here if notebook 01 did not run. Every source label space is collapsed to `fish`.

In [ ]:
# @title 1a · Dataset + splitsfrom src.data.split import build_detection_splitsfrom src.data.discovery import build_single_class_yolodet_dir = cfg.paths.data_interim / "detection_single_class"if not (det_dir / "images").exists():    print(build_single_class_yolo(cfg, det_dir))split_dir = cfg.paths.data_splits / "detection"if not (split_dir / "train_images.txt").exists():    print(build_detection_splits(sorted((det_dir / "images").glob("*")), split_dir,                                  seed=cfg.seed))data_yaml = det_dir / "data.yaml"print(data_yaml.read_text())

## 2 · Train the detector zooYOLO11 (n/s/m), YOLOv12-s and RT-DETR-l, all from **pretrained checkpoints**, single class.Each model is its own resumable stage: an interrupted run continues from `last.pt`(Ultralytics `resume=True`) instead of restarting.

In [ ]:
# @title 2a · Train (or resume) every configured detectorimport json, shutilfrom pathlib import Pathfrom src.utils.pretrained import ultralytics_model, PretrainedDownloadErrorresults = {}for name in cfg.raw["detector"]["models"]:    weights = cfg.raw["pretrained"]["detectors"][name]    key = f"detector.train.{name}"    run_dir = Path(cfg.paths.runs) / f"det_{name}"    best = run_dir / "weights" / "best.pt"    last = run_dir / "weights" / "last.pt"    if tr.is_done(key):        print(f"{name}: already trained ->", tr.get(key).metrics)        results[name] = tr.get(key).metrics        continue    with stage(tr, key) as st:        if st.skip:            continue        model = ultralytics_model(weights)          # raises if weights are unavailable        resume = bool(last.exists() and cfg.raw["resume"]["enabled"])        print(f"--- {name}: {'RESUMING from ' + str(last) if resume else 'fresh start'}")        train_kwargs = dict(            data=str(data_yaml), epochs=cfg.raw["detector"]["epochs"],            imgsz=cfg.raw["detector"]["imgsz"], batch=cfg.raw["detector"]["batch"],            patience=cfg.raw["detector"]["patience"], project=str(cfg.paths.runs),            name=f"det_{name}", exist_ok=True, seed=cfg.seed, amp=cfg.raw.get("amp", True),            plots=True, cos_lr=True, close_mosaic=10)        aug = cfg.raw["detector"]["augment"]        train_kwargs.update({k: aug[k] for k in ("mosaic", "hsv_h", "hsv_s", "hsv_v", "fliplr")                             if k in aug})        if resume:            train_kwargs["resume"] = True        model.train(**train_kwargs)        assert best.exists(), f"{name}: training finished but no best.pt"        dest = Path(cfg.paths.checkpoints) / f"det_{name}_best.pt"        shutil.copy2(best, dest)        metrics = {"checkpoint": str(dest), "weights_origin": weights}        results[name] = metrics        st.out(dest, run_dir / "results.csv")        st.checkpoint(dest)        st.metric(**{f"best_pt": 1})        print(f"{name} done ->", dest)json.dumps(results, indent=2)

### Optional: extra underwater augmentationsSpec 6 asks for motion blur, low light and turbidity. Ultralytics handles mosaics/HSV;these three are added as a dataset-level pre-pass into a shadow copy of the trainingimages (kept separate so the evaluation set is never altered).

In [ ]:
# @title 2b · Turbidity / low-light / motion-blur augment pass (train split only)import cv2, numpy as np, randomfrom pathlib import Pathkey = "detector.augment_pass"aug_dir = cfg.paths.data_interim / "detection_augmented"if tr.is_done(key):    print("resumed:", tr.get(key).metrics)else:    with stage(tr, key) as st:        if not st.skip:            rng = random.Random(cfg.seed)            train_list = [Path(p) for p in                          (split_dir / "train_images.txt").read_text().split("\n") if p]            aug_root = aug_dir / "images"; aug_root.mkdir(parents=True, exist_ok=True)            (aug_dir / "labels").mkdir(parents=True, exist_ok=True)            made = 0            for p in train_list:                img = cv2.imread(str(p))                if img is None:                    continue                if rng.random() < 0.5:                    h, w = img.shape[:2]                    out = img.astype(np.float32)                    if rng.random() < 0.4:      # turbidity haze                        out = out * 0.75 + np.array([120, 140, 135], dtype=np.float32) * 0.25                    if rng.random() < 0.4:      # low light                        out = (out / 255.0) ** 1.8 * 255.0                    if rng.random() < 0.3:      # motion blur                        k = rng.choice([5, 9, 13])                        kernel = np.zeros((k, k)); kernel[k // 2, :] = 1.0 / k                        out = cv2.filter2D(out, -1, kernel)                    out = np.clip(out, 0, 255).astype(np.uint8)                    cv2.imwrite(str(aug_root / p.name), out)                    lab = det_dir / "labels" / f"{p.stem}.txt"                    if lab.exists():                        (aug_dir / "labels" / f"{p.stem}.txt").write_bytes(lab.read_bytes())                    made += 1            st.metric(augmented_images=made)            print(f"augmented {made} training images -> {aug_dir}")

## 3 · Evaluate on the held-out detection split (own COCO-style evaluator)Precision / recall / F1 at the best-F1 confidence, mAP@50, mAP@50-95, PR curve andF1-confidence curve — the same evaluator for YOLO11, YOLOv12 and RT-DETR, so thecomparison is apples-to-apples.

In [ ]:
# @title 3a · Score every trained detector on the val splitimport numpy as np, pandas as pdfrom pathlib import Pathfrom src.utils.pretrained import ultralytics_modelfrom src.eval.detection_metrics import evaluate_detection, save_detection_metrics, plot_pr_and_f1def load_gt(img_paths, labels_dir):    out = []    for p in img_paths:        lab = Path(labels_dir) / f"{Path(p).stem}.txt"        boxes = []        if lab.exists():            for line in lab.read_text().splitlines():                parts = line.split()                if len(parts) >= 5:                    _, xc, yc, bw, bh = parts[:5]                    boxes.append([float(xc), float(yc), float(bw), float(bh)])        out.append(np.array(boxes, dtype=float).reshape(-1, 4))    return outdef to_xyxy(norm_boxes, w, h):    if len(norm_boxes) == 0:        return np.zeros((0, 4))    b = norm_boxes.copy()    b[:, 0] = (b[:, 0] - b[:, 2] / 2) * w    b[:, 1] = (b[:, 1] - b[:, 3] / 2) * h    b[:, 2] = (b[:, 0] + b[:, 2] * w); b[:, 3] = (b[:, 1] + b[:, 3] * h)    return bval_imgs = [Path(p) for p in (split_dir / "val_images.txt").read_text().split("\n") if p]gt_norm = load_gt(val_imgs, det_dir / "labels")all_metrics = {}for name in cfg.raw["detector"]["models"]:    ckpt = Path(cfg.paths.checkpoints) / f"det_{name}_best.pt"    if not ckpt.exists():        print(f"{name}: no checkpoint, skipped"); continue    key = f"detector.val.{name}"    if tr.is_done(key):        all_metrics[name] = tr.get(key).metrics        print(f"{name}: resumed", all_metrics[name]); continue    with stage(tr, key) as st:        if st.skip:            continue        model = ultralytics_model(str(ckpt))        preds, gts = [], []        for p, gt in zip(val_imgs, gt_norm):            import cv2            img = cv2.imread(str(p))            h, w = img.shape[:2]            r = model.predict(str(p), conf=0.001, iou=0.6, verbose=False)[0]            preds.append({"boxes": r.boxes.xyxy.cpu().numpy() if r.boxes is not None                          else np.zeros((0, 4)),                          "scores": r.boxes.conf.cpu().numpy() if r.boxes is not None                          else np.zeros(0)})            gts.append({"boxes": to_xyxy(gt, w, h)})        m = evaluate_detection(preds, gts)        save_detection_metrics(m, cfg.paths.metrics, f"detection_{name}",                               meta={"dataset": sorted(det_dir.glob("data.yaml"))[0].parent.name,                                     "split": "val", "n_images": len(val_imgs),                                     "weights": cfg.raw["pretrained"]["detectors"][name],                                     "seed": cfg.seed})        all_metrics[name] = {k: v for k, v in m.items() if isinstance(v, (int, float))}        st.metric(**all_metrics[name])        print(name, all_metrics[name])display(pd.DataFrame(all_metrics).T)

## 4 · Comparison table + curves

In [ ]:
# @title 4a · Write the model-comparison table and the PR/F1 figureimport json, pandas as pdcomp = pd.DataFrame(all_metrics).Tcomp.index.name = "model"comp.to_csv(cfg.paths.metrics / "detection_model_comparison.csv")if len(all_metrics):    curves = [json.loads((cfg.paths.metrics / f"detection_{n}.json").read_text())["metrics"]              for n in all_metrics]    plot_pr_and_f1(curves, list(all_metrics),                   cfg.paths.figures / "detection_pr_f1.png")display(comp[[c for c in comp.columns if c in              ("precision_at_best_f1", "recall_at_best_f1", "f1_at_best_f1",               "map50", "map50_95")]])

## 5 · Export detector crops for the classifier (cross-domain ablation input)

In [ ]:
# @title 5a · Crop classification images with the best detectorimport cv2, numpy as np, pandas as pdfrom pathlib import Pathfrom src.utils.pretrained import ultralytics_modelkey = "detector.export_crops"crop_dir = cfg.paths.data_interim / "detector_crops"if tr.is_done(key):    print("resumed:", tr.get(key).metrics)else:    with stage(tr, key) as st:        if not st.skip:            best_name = max(all_metrics, key=lambda n: all_metrics[n].get("map50", 0))            model = ultralytics_model(str(Path(cfg.paths.checkpoints) / f"det_{best_name}_best.pt"))            approved = pd.read_csv(cfg.paths.data_curated / "approved_labels.csv")            crop_dir.mkdir(parents=True, exist_ok=True)            rows = []            for _, r in approved.groupby("dataset_label").head(300).iterrows():                res = model.predict(r.path, conf=0.25, verbose=False)                if not res or res[0].boxes is None or len(res[0].boxes) == 0:                    continue                b = res[0].boxes.xyxy.cpu().numpy()[0]                rows.append({"path": r.path, "dataset_label": r.dataset_label,                             "x1": float(b[0]), "y1": float(b[1]),                             "x2": float(b[2]), "y2": float(b[3]),                             "score": float(res[0].boxes.conf.cpu().numpy()[0])})            pd.DataFrame(rows).to_csv(crop_dir / "crops.csv", index=False)            st.out(crop_dir / "crops.csv")            st.metric(crops=int(len(rows)), detector=best_name)            print(f"{len(rows)} crops from {best_name}")

### Next`04_classifier.ipynb` — species classifier on GT and on detector crops.The detector **test** split is evaluated once, in notebook 04's final comparison run, not here.